# 03b - Diagnostic de dependance a la carte

Ce diagnostic utilise exclusivement la zone development de fraudTrain. Il compare le meme XGBoost fixe avec trois schemas, sans tuning ni sous-echantillonnage. Il ne constitue pas le benchmark des cinq modeles.

## Regle fixee avant execution

PR-AUC signifie ici average precision. Si la moyenne des 5 plis aleatoires moins celle des 5 plis groupes par cc_num est strictement superieure a 0,02, la fuite est confirmee **au sens de cette regle operationnelle** et StratifiedGroupKFold est propose pour le benchmark. Sinon, StratifiedKFold reste propose. Ce diagnostic n'est pas une preuve causale d'une fuite d'identite.

Le controle temporel utilise les premiers 80 % du development pour apprendre et les 20 % suivants pour mesurer. Le resultat temporel ne modifie pas cette regle.

Configuration approuvee : 100 arbres, profondeur 3, learning_rate=0.1, seed=42, hist CPU, 1 thread, aucune ponderation ni sous-echantillonnage. Les deux CV ont 5 plis et shuffle=True. Le preprocessing est re-ajuste uniquement sur l'entrainement de chaque pli. Le cc_num et le trans_num ne sont jamais des features du modele.

Un passage au mode groupe modifierait la section 4.7 du document P2 et doit etre valide avec l'equipe avant T4.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "ml/src").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Racine du projet introuvable.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "ml/src"))
from validation.diagnostic import MODEL_PARAMS, GAP_THRESHOLD, decision_from_scores
print(MODEL_PARAMS)
print("Seuil predeclare :", GAP_THRESHOLD)

## Executer le diagnostic

Ce lancement entraine onze petits XGBoost successifs sur development uniquement. Le manifeste contenant la regle, les graines, les versions et le hash source est ecrit avant le premier ajustement. Les resultats sont conserves dans reports ; aucune etiquette de calibration n'est consultee par ce diagnostic.

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "ml/scripts/run_leakage_diagnostic.py")],
    cwd=PROJECT_ROOT, check=True,
)

## Structure des cartes

La proximite compte une fraude si une autre fraude de la meme carte la precede OU la suit de strictement moins de 24 h. Ce comptage sert au diagnostic ; il ne cree aucune feature d'historique pour le serving.

In [ ]:
summary = json.loads((PROJECT_ROOT / "reports/leakage_diagnostic_summary.json").read_text())
display(pd.DataFrame([summary["structure"]]))

## Scores par pli et comparaison

Les deux moyennes CV donnent le meme poids aux cinq plis. Le schema temporel n'a qu'un maintien 80/20. Toutes les valeurs sont lues depuis l'execution reelle.

In [ ]:
folds = pd.read_csv(PROJECT_ROOT / "reports/leakage_diagnostic_folds.csv")
display(folds)
display(folds.groupby("scheme")["pr_auc"].agg(["count", "mean", "std"]))
assert len(folds) == 11
assert folds.loc[folds["scheme"].eq("card_grouped"), "shared_cards"].eq(0).all()

## Conclusion calculee selon la regle predeclaree

In [ ]:
means = folds.groupby("scheme")["pr_auc"].mean()
decision = decision_from_scores(means["random_stratified"], means["card_grouped"])
assert decision["leakage_confirmed_by_rule"] == summary["decision"]["leakage_confirmed_by_rule"]
display(Markdown(
    f"PR-AUC aleatoire : **{means['random_stratified']:.6f}** ; "
    f"par carte : **{means['card_grouped']:.6f}** ; "
    f"temporelle : **{means['temporal_80_20']:.6f}**.\n\n"
    f"Ecart aleatoire - par carte : **{decision['gap_random_minus_grouped']:.6f}** ; "
    f"seuil : **{GAP_THRESHOLD}**. Fuite confirmee selon la regle : "
    f"**{decision['leakage_confirmed_by_rule']}**. Schema propose : "
    f"**{decision['proposed_benchmark_cv']}**.\n\n"
    "Arret avant T4 : examiner ces resultats avec P2 avant toute implementation du benchmark."
))